# Edge-AI Benchmark — Accuracy + Latency Sweep

Chạy inference qua một thư mục `samples/<class_name>/*.jpg` rồi tổng hợp:
- Accuracy / class
- Latency p50 / p95 / mean
- Confusion matrix

Notebook này tái dùng `EdgeAIOverlay` từ [edge_ai](../edge_ai/) — **chạy `inference_demo.ipynb` 1 lần trước** để confirm pipeline OK.


## 0. Setup


In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import allocate

sys.path.insert(0, str(Path.cwd().parent))
from edge_ai import EdgeAIOverlay, load_meta, preprocess_image, load_weights_blob, gap_argmax

PROJECT_ROOT = Path("/home/ubuntu/edgeai")
BITSTREAM    = PROJECT_ROOT / "hw/artifacts/kria_soc_wrapper.bit"
FIRMWARE     = PROJECT_ROOT / "firmware/firmware.bin"
WEIGHTS      = PROJECT_ROOT / "training/export/vgg-tiny_cats_dogs.weights.bin"
SAMPLES_DIR  = PROJECT_ROOT / "samples"        # SAMPLES_DIR/<label>/*.jpg


## 1. One-time setup: overlay + firmware + weights


In [ ]:
overlay = EdgeAIOverlay(str(BITSTREAM))
meta    = load_meta(str(WEIGHTS))
fpga    = meta["fpga"]

overlay.clear_shared_regs()
overlay.load_firmware(str(FIRMWARE))

wbytes = load_weights_blob(str(WEIGHTS))
wbuf = allocate(shape=(len(wbytes),), dtype=np.uint8)
wbuf[:] = np.frombuffer(wbytes, dtype=np.uint8); wbuf.flush()
overlay.set_weights_addr(wbuf.physical_address)

pp_size = fpga["max_tensor_bytes"]
buf_a = allocate(shape=(pp_size,), dtype=np.int8)
buf_b = allocate(shape=(pp_size,), dtype=np.int8)
overlay.set_dataset_id(meta["dataset_id"])

print(f"Ready. Labels={meta['labels']}  layers={fpga['num_layers']}")


## 2. Single-image inference helper

Tách logic kick/poll/decode thành 1 hàm để loop qua dataset.


In [ ]:
def infer_one(img_path: Path) -> tuple[int, float, float]:
    """Run inference on a single image. Returns (cls, conf_pct, latency_ms)."""
    ifm = preprocess_image(str(img_path), meta)
    flat = ifm.reshape(-1)
    buf_a[:flat.size] = flat
    buf_a.flush()
    overlay.set_io_buffers(buf_a.physical_address, buf_b.physical_address)

    t0 = time.perf_counter()
    overlay.kick()
    overlay.poll_done(timeout_s=5.0)
    latency_ms = (time.perf_counter() - t0) * 1000
    overlay.reset_cmd()

    final_buf = buf_a if fpga["final_ofm_buf_idx"] == 0 else buf_b
    final_buf.invalidate()
    H, W, C = fpga["final_ofm_shape"][-3:]
    ofm = np.frombuffer(final_buf[:H*W*C], dtype=np.int8).reshape(H, W, C)
    cls, conf, _ = gap_argmax(ofm, fpga["final_ofm_scale"], fpga["final_ofm_zp"])
    return cls, conf, latency_ms


## 3. Sweep dataset

Cây thư mục mong đợi:
```
samples/
├── cat/   *.jpg
└── dog/   *.jpg
```
Tên folder = label, phải khớp `meta['labels']`.


In [ ]:
records = []      # (true_cls, pred_cls, conf, latency_ms, path)
labels  = meta["labels"]
for true_idx, name in enumerate(labels):
    folder = SAMPLES_DIR / name
    if not folder.exists():
        print(f"[skip] {folder} not found"); continue
    for img in sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png")):
        cls, conf, lat = infer_one(img)
        records.append((true_idx, cls, conf, lat, img.name))

print(f"Inferred {len(records)} images")


## 4. Aggregate metrics


In [ ]:
if not records:
    print("No records — populate samples/ first.")
else:
    arr = np.array(records, dtype=object)
    true = arr[:, 0].astype(int)
    pred = arr[:, 1].astype(int)
    conf = arr[:, 2].astype(float)
    lat  = arr[:, 3].astype(float)

    acc = (true == pred).mean() * 100
    print(f"Accuracy : {acc:.2f}% ({(true==pred).sum()}/{len(true)})")
    print(f"Latency  : mean={lat.mean():.2f} ms  p50={np.median(lat):.2f}  p95={np.percentile(lat, 95):.2f}")

    # Confusion matrix
    n = len(labels)
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cm, cmap="Blues"); axes[0].set_title("Confusion matrix")
    axes[0].set_xticks(range(n)); axes[0].set_yticks(range(n))
    axes[0].set_xticklabels(labels); axes[0].set_yticklabels(labels)
    axes[0].set_xlabel("pred"); axes[0].set_ylabel("true")
    for i in range(n):
        for j in range(n):
            axes[0].text(j, i, cm[i, j], ha="center", va="center")
    axes[1].hist(lat, bins=20); axes[1].set_title("Latency distribution (ms)")
    axes[1].set_xlabel("ms"); axes[1].set_ylabel("count")
    plt.tight_layout(); plt.show()


## 5. Cleanup


In [ ]:
for b in (wbuf, buf_a, buf_b):
    b.freebuffer()
print("Done.")
